# 💾 Project 8.3 — LRU Cache for Sensor Readings
**DSA for Mechatronics · Week 08 — Hash Tables & Dictionaries**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib
from collections import Counter, defaultdict, OrderedDict
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A SCADA system caches the latest sensor readings in RAM.
The cache has a fixed capacity — when it's full and a new sensor is read,
the **Least Recently Used** entry is evicted to make room.

An **LRU Cache** is implemented with two data structures:
- A **hash map** for O(1) key lookup
- A **doubly-linked list** keeping insertion/access order (most recent at tail)

When a key is accessed or updated, it moves to the tail.
When the cache is full, the head (oldest) is evicted.

You will build it from scratch, then cross-check with Python's `OrderedDict`.


## Step 1 — LRU cache from scratch (doubly-linked list + hash map)

In [ ]:
# ── Personalised parameters ───────────────────────────────────────
CACHE_CAP   = random.randint(6, 12)
N_OPS       = random.randint(35, 55)
SENSOR_POOL = [f"S{i:03d}" for i in range(1, 20)]
OP_TYPES    = ["GET"] * 40 + ["PUT"] * 60

class DLLNode:
    """Node in doubly-linked list: stores key + value."""
    def __init__(self, key=None, val=None):
        self.key  = key
        self.val  = val
        self.prev = None
        self.next = None

class LRUCache:
    """
    LRU Cache with O(1) get and put.

    Design:
      - self.cache: dict  key → DLLNode
      - Doubly-linked list between dummy head (oldest) and tail (newest)
      - get(key)   → move node to tail, return value
      - put(key,v) → add/update at tail; if over capacity, evict head.next
    """
    def __init__(self, capacity):
        self.capacity = capacity
        self.cache    = {}
        # dummy sentinels avoid edge cases
        self.head = DLLNode()   # oldest end
        self.tail = DLLNode()   # newest end
        self.head.next = self.tail
        self.tail.prev = self.head
        self.evictions = 0
        self.hits      = 0
        self.misses    = 0

    # ── DLL helpers ──────────────────────────────────────────────
    def _remove(self, node):
        """Unlink node from list."""
        node.prev.next = node.next
        node.next.prev = node.prev

    def _add_to_tail(self, node):
        """Insert node just before dummy tail (most recent position)."""
        node.prev        = self.tail.prev
        node.next        = self.tail
        self.tail.prev.next = node
        self.tail.prev   = node

    # ── public API ───────────────────────────────────────────────
    def get(self, key):
        """O(1) lookup; moves node to most-recent (tail) position."""
        if key in self.cache:
            node = self.cache[key]
            self._remove(node)
            self._add_to_tail(node)
            self.hits += 1
            return node.val
        self.misses += 1
        return -1

    def put(self, key, value):
        """O(1) insert/update. Evicts LRU entry if over capacity."""
        if key in self.cache:
            node = self.cache[key]
            node.val = value
            self._remove(node)
            self._add_to_tail(node)
        else:
            if len(self.cache) >= self.capacity:
                # evict LRU: node right after dummy head
                lru = self.head.next
                self._remove(lru)
                del self.cache[lru.key]
                self.evictions += 1
            node = DLLNode(key, value)
            self.cache[key] = node
            self._add_to_tail(node)

    def size(self):
        return len(self.cache)

    def order(self):
        """Return keys from LRU to MRU (head.next → tail.prev)."""
        result, node = [], self.head.next
        while node is not self.tail:
            result.append(node.key)
            node = node.next
        return result

print(f"LRU Cache parameters:")
print(f"  Capacity   : {CACHE_CAP}")
print(f"  Operations : {N_OPS}")
print(f"  Sensor pool: {len(SENSOR_POOL)} sensors")
print()

lru = LRUCache(CACHE_CAP)

# Generate operation sequence
ops = []
temp_vals = {}
for _ in range(N_OPS):
    op = random.choice(OP_TYPES)
    sensor = random.choice(SENSOR_POOL)
    if op == "PUT":
        val = round(random.uniform(20.0, 120.0), 1)
        ops.append(("PUT", sensor, val))
        temp_vals[sensor] = val
    else:
        ops.append(("GET", sensor, None))

# Execute and log
eviction_log = []
print(f"  {'#':>3}  {'Op':<4}  {'Key':<6}  {'Val':>8}  {'Result':>8}  {'Cache order (LRU→MRU)'}")
print(f"  {'─'*3}  {'─'*4}  {'─'*6}  {'─'*8}  {'─'*8}  {'─'*30}")
for i, (op, key, val) in enumerate(ops):
    pre_size = lru.size()
    if op == "PUT":
        lru.put(key, val)
        result = f"stored"
    else:
        result_val = lru.get(key)
        result = f"{result_val}" if result_val != -1 else "MISS"
    if lru.size() < pre_size + (1 if op=="PUT" else 0):
        eviction_log.append(i)
    order = lru.order()
    order_str = "→".join(order)
    print(f"  {i:3d}  {op:<4}  {key:<6}  {str(val) if val else '—':>8}  {result:>8}  {order_str}")


## Step 2 — Cross-check with OrderedDict

In [ ]:
class LRUOrderedDict:
    """Reference implementation using Python's OrderedDict."""
    def __init__(self, capacity):
        self.capacity = capacity
        self.cache    = OrderedDict()
        self.evictions = 0

    def get(self, key):
        if key not in self.cache: return -1
        self.cache.move_to_end(key)   # mark as most recently used
        return self.cache[key]

    def put(self, key, value):
        if key in self.cache:
            self.cache.move_to_end(key)
            self.cache[key] = value
        else:
            if len(self.cache) >= self.capacity:
                self.cache.popitem(last=False)   # evict LRU (first item)
                self.evictions += 1
            self.cache[key] = value

ref = LRUOrderedDict(CACHE_CAP)
for op, key, val in ops:
    if op == "PUT": ref.put(key, val)
    else:           ref.get(key)

# Compare final states
custom_order = lru.order()
ref_order    = list(ref.cache.keys())
match        = custom_order == ref_order

print(f"Cross-check against OrderedDict reference:")
print(f"  Custom LRU order : {custom_order}")
print(f"  Reference order  : {ref_order}")
print(f"  Orders match     : {'✅ YES' if match else '❌ NO'}")
print()
print(f"Cache statistics:")
print(f"  Hits             : {lru.hits}")
print(f"  Misses           : {lru.misses}")
hit_rate = lru.hits / (lru.hits + lru.misses) * 100 if (lru.hits + lru.misses) else 0
print(f"  Hit rate         : {hit_rate:.1f}%")
print(f"  Evictions        : {lru.evictions}")
print(f"  Final cache size : {lru.size()}")


## Step 3 — Frequency analysis of sensor accesses

In [ ]:
# Count how often each sensor was accessed (PUT or GET)
access_counts = Counter(key for _, key, _ in ops)
put_counts    = Counter(key for op, key, _ in ops if op == "PUT")
get_counts    = Counter(key for op, key, _ in ops if op == "GET")

print(f"Top sensor access frequency ({N_OPS} total operations):")
print(f"  {'Sensor':<8}  {'Total':>7}  {'PUTs':>6}  {'GETs':>6}  Bar")
print(f"  {'─'*8}  {'─'*7}  {'─'*6}  {'─'*6}  {'─'*20}")
for sensor, count in access_counts.most_common(10):
    bar = "█" * count
    print(f"  {sensor:<8}  {count:7d}  {put_counts.get(sensor,0):6d}  "
          f"{get_counts.get(sensor,0):6d}  {bar}")

most_accessed = access_counts.most_common(1)[0]
least_accessed = access_counts.most_common()[-1]
print(f"\n  Most accessed  : {most_accessed[0]} ({most_accessed[1]} times)")
print(f"  Least accessed : {least_accessed[0]} ({least_accessed[1]} time(s))")
print(f"  Unique sensors accessed: {len(access_counts)}")


## Step 4 — Eviction pattern analysis

In [ ]:
# Replay operations tracking evictions and cache state snapshots
lru2 = LRUCache(CACHE_CAP)
snapshot_interval = max(1, N_OPS // 8)
snapshots = []

for i, (op, key, val) in enumerate(ops):
    if op == "PUT":
        lru2.put(key, val)
    else:
        lru2.get(key)
    if i % snapshot_interval == 0 or i == N_OPS - 1:
        snapshots.append((i, lru2.size(), lru2.evictions, list(lru2.order())))

print(f"Cache size & evictions over time (every {snapshot_interval} ops):")
print(f"  {'Op #':>5}  {'Size':>5}  {'Evictions':>10}  LRU→MRU snapshot")
print(f"  {'─'*5}  {'─'*5}  {'─'*10}  {'─'*30}")
for op_idx, size, evict, order in snapshots:
    print(f"  {op_idx:5d}  {size:5d}  {evict:10d}  {'→'.join(order)}")

print()
print(f"Final cache state:")
final_order = lru2.order()
print(f"  LRU → MRU: {' → '.join(final_order)}")
print(f"  Size      : {lru2.size()} / {CACHE_CAP}")
print(f"  Hit rate  : {lru2.hits/(lru2.hits+lru2.misses)*100:.1f}%")
print(f"  Evictions : {lru2.evictions}")


## 📋 Final Report

In [ ]:
W = 56
print("╔" + "═"*W + "╗")
print("║" + " LRU CACHE FOR SENSOR READINGS — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<26}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<26}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  CACHE PARAMETERS" + " "*(W-18) + "║")
print(f"║  {'Cache capacity':<26}: {CACHE_CAP:<26}║")
print(f"║  {'Operations':<26}: {N_OPS:<26}║")
print(f"║  {'Sensor pool size':<26}: {len(SENSOR_POOL):<26}║")
print("╠" + "═"*W + "╣")
print("║  CACHE PERFORMANCE" + " "*(W-19) + "║")
print(f"║  {'Hits':<26}: {lru.hits:<26}║")
print(f"║  {'Misses':<26}: {lru.misses:<26}║")
print(f"║  {'Hit rate':<26}: {hit_rate:.1f}%{'':<22}║")
print(f"║  {'Evictions':<26}: {lru.evictions:<26}║")
print("╠" + "═"*W + "╣")
print("║  VERIFICATION" + " "*(W-14) + "║")
print(f"║  {'OrderedDict match':<26}: {'YES ✅' if match else 'NO ❌':<26}║")
print(f"║  {'Most accessed sensor':<26}: {most_accessed[0]} ({most_accessed[1]} ops){'':<10}║")
print(f"║  {'Unique sensors touched':<26}: {len(access_counts):<26}║")
print("╠" + "═"*W + "╣")
print("║  FINAL CACHE STATE" + " "*(W-19) + "║")
final_str = "→".join(final_order)[:46]
print(f"║  {'LRU→MRU order':<26}: {final_str:<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about the system?

*Your answer here:*

---

**Q2 — Which hash table concept did you find most important, and why?**
Pick one technique (collision resolution, load factor, LRU cache, rolling hash…) and explain what problem it solves.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
